# Experimento VQE — versão 20.6 focada

## Varredura individual em 100 vetores selecionados pela máscara — sem COBYLA

Este notebook mantém somente a preparação necessária e o experimento de varredura individual. Ele utiliza o arquivo `merge.pkl`, reconstrói o Hamiltoniano e o ansatz e seleciona **100 vetores completos** entre os melhores candidatos da máscara.

A intervenção continua direta:

$$
\theta^{(a)} \longrightarrow U(\theta^{(a)}) \longrightarrow |\psi(\theta^{(a)})\rangle,
$$

sem COBYLA e sem reotimização dos demais parâmetros.

### Objetivos desta versão

1. selecionar 100 vetores completos do conjunto superior produzido pela máscara;
2. variar somente um componente $\theta_j$ por vez em cada vetor;
3. manter todos os demais componentes do mesmo vetor exatamente fixos;
4. conservar a probabilidade bruta $P(\mathcal{X}_{\mathrm{opt}})$, sem normalização min–max;
5. marcar os 100 valores originais de cada $\theta_j$;
6. sobrepor as 100 curvas de cada parâmetro e calcular envelopes de distribuição;
7. reunir os valores originais canônicos de todos os vetores em um único gráfico;
8. testar empiricamente se os parâmetros `CRY`, estruturalmente varridos em $4\pi$, já repetem a probabilidade após $2\pi$;
9. confirmar que o conjunto de ativos de cada $\theta_j$ é fixado pelo bloco lógico do ansatz e não muda com o vetor.

Todos os gráficos abaixo usam o mesmo Hamiltoniano e o mesmo ansatz. O que muda entre as 100 curvas é somente o vetor de parâmetros escolhido pela máscara.

## Como interpretar o circuito antes dos testes

O circuito começa aplicando portas `X` em $k$ qubits. Isso prepara **um estado-base inicial com peso de Hamming $k$**; não significa que a solução ótima já foi inserida no circuito.

Depois, os blocos parametrizados redistribuem a amplitude entre estados que mantêm a cardinalidade:

- `CY`: bloco lógico de **dois qubits**, decomposto com uma rotação controlada `CRY`;
- `CCY`: bloco lógico de **três qubits**, decomposto com `RY` e `CCX`;
- `RY`: operação primitiva de um qubit;
- `CX`: operação primitiva de dois qubits;
- `CCX`: operação primitiva de três qubits.

Portanto, não é correto dizer que toda porta é simplesmente uma junção de dois spins. O Hamiltoniano do portfólio é diagonal e contém termos de um e dois corpos, `Z` e `ZZ`, enquanto o **ansatz** usa blocos de dois e três qubits para navegar no subespaço de Dicke.

O índice $j$ em $\theta_j$ não deve ser fornecido isoladamente ao Transformer como significado físico. O objeto transferível é a descrição estrutural:

$$
(\text{tipo de bloco},\; \text{qubits/ativos},\; \text{distância},\;
\text{posição},\; \text{período},\; \text{termos do Hamiltoniano tocados}).
$$


### Célula 1 — Importações, parâmetros e pastas do experimento

**Em termos simples:** esta célula configura somente o que é necessário para selecionar 100 vetores e executar as varreduras individuais.

**O que é configurado:**

- os parâmetros financeiros $q$, $r_f$ e a cardinalidade $k=4$;
- a máscara dos 10% melhores;
- a seleção final de exatamente 100 vetores completos;
- os índices $\theta_j$ testados;
- 201 pontos regulares por período, além do valor original inserido exatamente;
- checkpoints por par `(vetor, theta)`, permitindo retomar uma execução interrompida;
- um diretório novo de saída, isolado das versões com um ou dez vetores.

A grade usa o período estrutural de cada porta: `RY` em $[0,2\pi]$ e `CRY` em $[0,4\pi]$. Uma auditoria posterior verifica se, para o observável de probabilidade, os dois ciclos de uma `CRY` são de fato diferentes.

In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO ÚNICA
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
import ast
import hashlib
import json
import math
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import spearmanr

NOTEBOOK_VERSION = "20.6-100-masked-vectors-sweep"
RANDOM_SEED = 42

# Parâmetros do problema financeiro.
Q_VALUE = 0.5
RISK_FREE = 0.0475
TARGET_K = 4
ENERGY_ATOL = 1e-8

# ------------------------------------------------------------
# ÚNICO CAMINHO DE ENTRADA
# Altere somente esta linha quando o merge.pkl estiver em outro local.
# ------------------------------------------------------------
MERGE_PKL = Path(r"C:\Users\Marlon_Kelly\Downloads\merge.pkl")

# A máscara conserva aproximadamente os 10% melhores vetores salvos.
TOP_FRACTION = 0.10

# Reavalia mais candidatos do que o número final para escolher 100 vetores
# usando as métricas exatas reconstruídas neste notebook.
MAX_EXACT_REEVALUATION = 300
N_ANCHORS = 100

# Parâmetros varridos individualmente em cada um dos 100 vetores.
ACTIVE_THETA_INDICES = [2, 14, 17, 19, 22, 25, 27]
DETAILED_THETA_INDICES = [17, 2, 14, 19, 22, 25, 27, 3, 24]

# 201 pontos regulares por período, mais o ponto original quando necessário.
# Isso controla o custo: 100 vetores x 9 thetas x aproximadamente 202 pontos.
SWEEP_POINTS_PER_PERIOD = 201
COMMON_PHASE_POINTS = 201

# Critério apenas para informar se uma curva é numericamente plana.
SWEEP_FLAT_ABS_TOL = 1e-10
SWEEP_FLAT_REL_TOL = 1e-8
PERIODICITY_ATOL = 1e-9
PERIODICITY_RTOL = 1e-7

# Checkpoints novos desta versão podem ser reutilizados para retomar a campanha.
REUSE_SWEEP_CHECKPOINTS = True

# Pasta nova e isolada: evita misturar curvas de versões anteriores.
OUTPUT_ROOT = Path("vqe_r") / "pipeline_v20_6_100_masked_vectors_sweep"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
DISTRIBUTION_DIR = OUTPUT_ROOT / "distributions"

for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR, DISTRIBUTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_version": NOTEBOOK_VERSION,
    "merge_pkl": str(MERGE_PKL),
    "target_k": TARGET_K,
    "q_value": Q_VALUE,
    "risk_free": RISK_FREE,
    "top_fraction": TOP_FRACTION,
    "n_selected_vectors": N_ANCHORS,
    "max_exact_reevaluation": MAX_EXACT_REEVALUATION,
    "active_theta_indices": ACTIVE_THETA_INDICES,
    "detailed_theta_indices": DETAILED_THETA_INDICES,
    "sweep_points_per_period": SWEEP_POINTS_PER_PERIOD,
    "reuse_sweep_checkpoints": REUSE_SWEEP_CHECKPOINTS,
    "optimizer_used": False,
    "cobyla_calls": 0,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("Saídas:", OUTPUT_ROOT.resolve())

# Parte I — carregar e auditar somente o `merge.pkl`

O carregamento não procura nomes alternativos em vários diretórios. Existe um único caminho configurado em `MERGE_PKL`.

A célula seguinte aceita `DataFrame`, lista de dicionários ou dicionário serializado, mas não reconstrói nem modifica o banco original.


### Célula 2 — Carregamento controlado do banco `merge.pkl`

**Em termos simples:** esta célula abre o único arquivo de entrada e o transforma em um `DataFrame` de trabalho.

Ela verifica se o caminho existe, aceita três formatos serializados (`DataFrame`, lista de registros ou dicionário) e interrompe a execução caso o arquivo esteja vazio ou tenha um tipo inesperado.

**Saída principal:** `merge_df`, que contém o banco original carregado em memória. O arquivo em disco não é modificado.


In [ ]:
# ============================================================
# 2. CARREGAMENTO ÚNICO DO merge.pkl
# ============================================================

# Resolve "~", converte para caminho absoluto e verifica o arquivo antes da leitura.
merge_path = MERGE_PKL.expanduser().resolve()
if not merge_path.is_file():
    raise FileNotFoundError(
        "merge.pkl não encontrado. Caminho configurado: "
        f"{merge_path}"
    )

# O pickle é lido uma única vez. As conversões seguintes ocorrem apenas em memória.
loaded_object = pd.read_pickle(merge_path)

# Padroniza diferentes formatos serializados para um único DataFrame.
if isinstance(loaded_object, pd.DataFrame):
    merge_df = loaded_object.copy()
elif isinstance(loaded_object, list):
    merge_df = pd.DataFrame(loaded_object)
elif isinstance(loaded_object, dict):
    merge_df = pd.DataFrame(loaded_object)
else:
    raise TypeError(
        "O merge.pkl deve conter DataFrame, lista de registros ou dicionário; "
        f"tipo encontrado: {type(loaded_object)}"
    )

if merge_df.empty:
    raise ValueError("O merge.pkl foi carregado, mas está vazio.")

print("Arquivo:", merge_path)
print("Shape:", merge_df.shape)
print("Colunas:", merge_df.columns.tolist())
display(merge_df.head())


### Célula 3 — Identificação das colunas e conversão dos dados

**Em termos simples:** bancos gerados em versões diferentes podem usar nomes diferentes para a mesma informação. Esta célula cria um mapa de aliases e identifica qual coluna representa retorno, covariância, vetor de parâmetros, energia, probabilidade e bitstring.

Também são definidas funções para converter conteúdos salvos como texto em objetos numéricos:

- `parse_tickers`: recupera os nomes dos ativos;
- `parse_vector`: transforma retornos e vetores $\theta$ em arrays;
- `parse_matrix`: reconstrói a matriz de covariância;
- `normalize_bitstring`: padroniza bitstrings para uma sequência de zeros e uns.

**Importante:** essa normalização ocorre apenas na cópia em memória. O `merge.pkl` original permanece intacto.


In [ ]:
# ============================================================
# 3. NORMALIZAÇÃO DO ESQUEMA SEM ALTERAR O ARQUIVO ORIGINAL
# ============================================================

# Cada chave representa um conceito do experimento; a lista contém nomes de
# coluna aceitos para esse mesmo conceito em versões diferentes do banco.
COLUMN_ALIASES = {
    "tickers": ["tickers", "assets", "asset_names"],
    "assets_return": ["assets_return", "assets_returns", "expected_returns", "mu"],
    "covariance": ["covariance", "covariance_matrix", "sigma"],
    "best_parameters": ["best_parameters", "theta", "theta_final"],
    "initial_point": ["initial_point", "initial_theta", "theta_initial"],
    "objective": ["objective_function_value", "energy", "final_energy"],
    "best_objective": ["best_objective_function_value", "exact_energy", "optimal_energy"],
    "p_best": [
        "p_exact_eval",
        "probability_best_answer",
        "probability_best_answer_shots",
        "p_best",
        "prob_best",
    ],
    "gap": ["gap_exact_eval", "energy_gap", "gap"],
    "dominant_bitstring": [
        "most_frequent_bitstring",
        "most_frequen_bitstring",
        "best_answer",
        "dominant_bitstring",
    ],
    "counts": ["counts", "measurement_counts"],
    "status": ["status"],
}


def first_existing_column(frame, aliases, required=False):
    """Retorna o primeiro alias realmente presente no DataFrame."""
    for name in aliases:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(f"Nenhuma das colunas obrigatórias foi encontrada: {aliases}")
    return None


# Resultado final do mapeamento: conceito lógico -> nome real no merge.pkl.
RESOLVED_COLUMNS = {
    key: first_existing_column(
        merge_df,
        aliases,
        required=key in {"tickers", "assets_return", "covariance", "best_parameters"},
    )
    for key, aliases in COLUMN_ALIASES.items()
}


def parse_serialized(value):
    """Converte texto serializado em lista, dicionário ou array quando possível."""
    if isinstance(value, (np.ndarray, list, tuple, dict, pd.Series, pd.Index, pd.DataFrame)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                pass
        cleaned = text.strip("[]()")
        arr = np.fromstring(cleaned.replace(",", " "), sep=" ")
        if arr.size:
            return arr
    return value


def parse_tickers(value):
    """Padroniza a lista de tickers e rejeita nomes vazios."""
    parsed = parse_serialized(value)
    if isinstance(parsed, str):
        items = [item.strip() for item in parsed.replace(";", ",").split(",")]
    elif isinstance(parsed, dict):
        items = list(parsed.keys())
    else:
        items = list(parsed)
    tickers = [str(item).strip().strip("'\"") for item in items]
    if not tickers or any(not item for item in tickers):
        raise ValueError(f"Tickers inválidos: {value}")
    return tickers


def parse_vector(value, tickers=None):
    """Converte um vetor salvo em texto, Series, dicionário ou lista para NumPy."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.Series):
        if tickers is not None and set(tickers).issubset(set(parsed.index.astype(str))):
            return parsed.reindex(tickers).to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        if tickers is not None and set(tickers).issubset(set(map(str, parsed.keys()))):
            return np.asarray([parsed[ticker] for ticker in tickers], dtype=float)
        return np.asarray(list(parsed.values()), dtype=float)
    return np.asarray(parsed, dtype=float).reshape(-1)


def parse_matrix(value, tickers=None):
    """Reconstrói uma matriz numérica e preserva a ordem dos tickers quando possível."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.DataFrame):
        if tickers is not None:
            return parsed.loc[tickers, tickers].to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        frame = pd.DataFrame(parsed)
        if tickers is not None and set(tickers).issubset(frame.index) and set(tickers).issubset(frame.columns):
            return frame.loc[tickers, tickers].to_numpy(dtype=float)
        return frame.to_numpy(dtype=float)
    array = np.asarray(parsed, dtype=float)
    if array.ndim == 1:
        n = int(round(np.sqrt(array.size)))
        if n * n != array.size:
            raise ValueError("Covariância unidimensional não forma uma matriz quadrada.")
        array = array.reshape(n, n)
    return array


def normalize_bitstring(value):
    """Remove prefixos e espaços, retornando apenas bitstrings binários válidos."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).replace(" ", "").replace("'", "").replace('"', "")
    if text.startswith("0b"):
        text = text[2:]
    return text if set(text).issubset({"0", "1"}) else None


print(json.dumps(RESOLVED_COLUMNS, indent=2, ensure_ascii=False))


### Célula 4 — Reconstrução do problema financeiro e auditoria do banco

**Em termos simples:** a primeira linha válida do banco é usada para recuperar os ativos, o vetor de retornos $\mu$ e a matriz de covariância $\Sigma$.

A covariância é explicitamente simetrizada:

$$
\Sigma_{\mathrm{sim}}=\frac{\Sigma+\Sigma^\mathsf{T}}{2}.
$$

Depois, uma amostra de até 200 linhas recebe uma impressão digital (`hash`). Se aparecer mais de um hash, o banco contém mais de um problema/Hamiltoniano e a execução é interrompida.

A célula também estima a cardinalidade pelos bitstrings salvos e cria `problem_summary_df`, com retorno, variância e conexão de risco de cada ativo.


In [ ]:
# ============================================================
# 4. EXTRAIR O HAMILTONIANO E AUDITAR CONSISTÊNCIA DO BANCO
# ============================================================

# Usa a primeira linha com um vetor theta válido como referência do problema.
first_valid_index = merge_df[
    merge_df[RESOLVED_COLUMNS["best_parameters"]].notna()
].index[0]
first_row = merge_df.loc[first_valid_index]

tickers = parse_tickers(first_row[RESOLVED_COLUMNS["tickers"]])
mu = parse_vector(first_row[RESOLVED_COLUMNS["assets_return"]], tickers=tickers)
sigma = parse_matrix(first_row[RESOLVED_COLUMNS["covariance"]], tickers=tickers)
# Corrige pequenas assimetrias numéricas sem alterar a parte simétrica do risco.
sigma = 0.5 * (sigma + sigma.T)

N_ASSETS = len(tickers)
if mu.shape != (N_ASSETS,):
    raise ValueError(f"Retornos com shape {mu.shape}; esperado {(N_ASSETS,)}.")
if sigma.shape != (N_ASSETS, N_ASSETS):
    raise ValueError(
        f"Covariância com shape {sigma.shape}; esperado {(N_ASSETS, N_ASSETS)}."
    )
if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Retornos ou covariância possuem valores não finitos.")

# A cardinalidade é inferida do bitstring salvo quando possível.
bit_col = RESOLVED_COLUMNS["dominant_bitstring"]
if bit_col is not None:
    saved_bits = merge_df[bit_col].map(normalize_bitstring).dropna()
    inferred_weights = saved_bits.map(lambda value: value.count("1"))
    inferred_k = int(inferred_weights.mode().iloc[0]) if not inferred_weights.empty else TARGET_K
else:
    inferred_k = TARGET_K

if inferred_k != TARGET_K:
    warnings.warn(
        f"A cardinalidade modal inferida foi k={inferred_k}; "
        f"o experimento está configurado para k={TARGET_K}."
    )

# Confirma que uma amostra do merge representa o mesmo problema.
def problem_fingerprint(row):
    """Cria um hash a partir de tickers, retornos e covariância de uma linha."""
    row_tickers = parse_tickers(row[RESOLVED_COLUMNS["tickers"]])
    row_mu = parse_vector(row[RESOLVED_COLUMNS["assets_return"]], tickers=row_tickers)
    row_sigma = parse_matrix(row[RESOLVED_COLUMNS["covariance"]], tickers=row_tickers)
    payload = (
        "|".join(row_tickers).encode("utf-8")
        + np.asarray(row_mu, dtype=np.float64).tobytes()
        + np.asarray(row_sigma, dtype=np.float64).tobytes()
    )
    return hashlib.sha256(payload).hexdigest()[:16]

# Uma amostra aleatória é suficiente para detectar mistura evidente de problemas,
# sem reler e converter necessariamente todas as linhas do banco.
sample_size = min(200, len(merge_df))
sampled_rows = merge_df.sample(sample_size, random_state=RANDOM_SEED)
fingerprints = sampled_rows.apply(problem_fingerprint, axis=1)
if fingerprints.nunique() != 1:
    raise RuntimeError(
        "O merge.pkl contém mais de um Hamiltoniano na amostra auditada. "
        "Este notebook 20.6 executa um problema por vez; filtre o merge antes de continuar."
    )

DATA_HASH = fingerprints.iloc[0]
min_cov_eigenvalue = float(np.linalg.eigvalsh(sigma).min())

# Tabela por ativo usada posteriormente como descrição estrutural do problema.
problem_summary_df = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS, dtype=int),
    "ticker": tickers,
    "return_sum": mu,
    "variance": np.diag(sigma),
    "risk_connection_abs": np.sum(np.abs(sigma), axis=1),
})

print("n =", N_ASSETS, "| k =", TARGET_K)
print("tickers =", tickers)
print("problem_hash =", DATA_HASH)
print("menor autovalor da covariância =", f"{min_cov_eigenvalue:.3e}")
display(problem_summary_df)


# Parte II — referência clássica e rigidez dos pares

## O que esta parte representa

Antes de analisar o circuito quântico, o notebook resolve exatamente o problema clássico. Como existem 10 ativos e o portfólio deve selecionar 4, o número total de soluções válidas é

$$
\binom{10}{4}=210.
$$

Isso significa que é possível avaliar todos os 210 portfólios e conhecer, sem aproximação:

- a energia mínima exata;
- todos os bitstrings ótimos;
- a distância energética entre decisões concorrentes;
- quais pares de ativos possuem decisões mais rígidas.

## Mínimos condicionados de cada par

Para cada par de ativos $(i,j)$, fixamos os valores de decisão $x_i=a$ e $x_j=b$, com $a,b\in\{0,1\}$. Em seguida, procuramos o melhor portfólio que respeita essas duas decisões e continua selecionando exatamente $k$ ativos:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_{\ell}x_{\ell}=k}}
E(x).
$$

Assim, cada par possui quatro energias condicionadas:

$$
E_{ij}^{00},\qquad
E_{ij}^{01},\qquad
E_{ij}^{10},\qquad
E_{ij}^{11}.
$$

Esses valores permitem medir quanto custa trocar a decisão de um ativo, dos dois ativos e quão separado está o melhor estado do par em relação ao segundo melhor. Mais adiante, essas métricas serão ligadas aos blocos quânticos que atuam sobre os mesmos qubits.


### Célula 5 — Solução clássica exata e rigidez dos pares de ativos

**Em termos simples:** para 10 ativos escolhendo exatamente 4, todos os portfólios válidos podem ser enumerados:

$$
N_{\mathrm{portfólios}}=\binom{10}{4}=210.
$$

A função objetivo avaliada para cada bitstring é

$$
E(x)=q\,x^\mathsf{T}\Sigma x-(1-q)\,\mu^\mathsf{T}x+r_f,
$$

com a restrição $\sum_i x_i=k$.

Para cada par de ativos $(i,j)$ e cada estado $a,b\in\{0,1\}$, é calculado o melhor portfólio condicionado:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_\ell x_\ell=k}}
E(x).
$$

Esses quatro mínimos permitem medir:

- `G_ij`: separação entre o melhor e o segundo melhor estado condicionado do par;
- gaps de trocar apenas $i$, apenas $j$ ou os dois;
- não aditividade da troca conjunta.

**Saídas principais:** `enumeration_df`, `asset_decision_df`, `pair_gap_df`, `exact_energy` e os bitstrings ótimos.


In [ ]:
# ============================================================
# 5. ENUMERAÇÃO CLÁSSICA EXATA E GAPS CONDICIONAIS
# ============================================================

PAIR_STATES = ("00", "01", "10", "11")


def portfolio_objective(x_binary):
    """Calcula E(x)=q*x.T*Sigma*x-(1-q)*mu.T*x+r_f para um portfólio binário."""
    x = np.asarray(x_binary, dtype=float).reshape(-1)
    return float(
        Q_VALUE * x @ sigma @ x
        - (1.0 - Q_VALUE) * mu @ x
        + RISK_FREE
    )


def bitstring_asset_order(x):
    """Converte o vetor binário para a ordem natural dos ativos."""
    return "".join(str(int(value)) for value in np.asarray(x, dtype=int))


def enumerate_portfolios(k_value=TARGET_K):
    """Enumera exatamente todas as combinações de k ativos entre N_ASSETS."""
    rows = []
    for selected_indices in combinations(range(N_ASSETS), int(k_value)):
        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected_indices)] = 1
        bits = bitstring_asset_order(x)
        rows.append({
            "x_asset_order": tuple(int(v) for v in x),
            "bitstring_asset_order": bits,
            "bitstring_qiskit_order": bits[::-1],
            "selected_assets": tuple(tickers[index] for index in selected_indices),
            "objective": portfolio_objective(x),
        })
    return pd.DataFrame(rows).sort_values(
        ["objective", "bitstring_asset_order"]
    ).reset_index(drop=True)


# Para n=10 e k=4, esta tabela possui C(10,4)=210 linhas.
enumeration_df = enumerate_portfolios(TARGET_K)
exact_energy = float(enumeration_df.iloc[0]["objective"])
optimal_mask = np.isclose(
    enumeration_df["objective"].to_numpy(dtype=float),
    exact_energy,
    atol=ENERGY_ATOL,
    rtol=0.0,
)
optimal_df = enumeration_df.loc[optimal_mask].copy()
exact_asset_bitstrings = sorted(optimal_df["bitstring_asset_order"].unique())
exact_qiskit_bitstrings = sorted(optimal_df["bitstring_qiskit_order"].unique())
exact_x = np.asarray([int(value) for value in exact_asset_bitstrings[0]], dtype=int)
energy_span = float(enumeration_df["objective"].max() - exact_energy)


def build_asset_decision_margins():
    """Mede o custo mínimo de inverter a decisão de cada ativo no ótimo clássico."""
    selected = np.flatnonzero(exact_x == 1)
    excluded = np.flatnonzero(exact_x == 0)
    rows = []
    for asset_index in range(N_ASSETS):
        alternatives = []
        if exact_x[asset_index] == 1:
            for replacement in excluded:
                trial = exact_x.copy()
                trial[asset_index] = 0
                trial[replacement] = 1
                alternatives.append(portfolio_objective(trial))
        else:
            for removed in selected:
                trial = exact_x.copy()
                trial[asset_index] = 1
                trial[removed] = 0
                alternatives.append(portfolio_objective(trial))
        margin = max(min(alternatives) - exact_energy, 0.0)
        rows.append({
            "asset_index": asset_index,
            "ticker": tickers[asset_index],
            "selected_exact": int(exact_x[asset_index]),
            "decision_margin": float(margin),
            "decision_margin_relative": float(margin / max(energy_span, ENERGY_ATOL)),
        })
    return pd.DataFrame(rows).sort_values("decision_margin", ascending=False)


asset_decision_df = build_asset_decision_margins()


def conditional_pair_best(i, j, state):
    """Retorna o melhor portfólio sob x_i=a e x_j=b para um estado ab."""
    a, b = int(state[0]), int(state[1])
    subset = enumeration_df.loc[
        enumeration_df["x_asset_order"].map(
            lambda x: int(x[i]) == a and int(x[j]) == b
        )
    ]
    if subset.empty:
        raise RuntimeError(f"Estado inviável para par {(i, j)}: {state}")
    return subset.iloc[0]


# Para cada par, calculamos E_00, E_01, E_10 e E_11 e derivamos os gaps.
pair_rows = []
for i, j in combinations(range(N_ASSETS), 2):
    states = {state: conditional_pair_best(i, j, state) for state in PAIR_STATES}
    ordered = sorted(PAIR_STATES, key=lambda state: (states[state]["objective"], state))
    best_state, second_state = ordered[:2]
    global_state = f"{exact_x[i]}{exact_x[j]}"
    a_star, b_star = map(int, global_state)
    flip_i = f"{1-a_star}{b_star}"
    flip_j = f"{a_star}{1-b_star}"
    flip_both = f"{1-a_star}{1-b_star}"
    delta_i = max(float(states[flip_i]["objective"] - exact_energy), 0.0)
    delta_j = max(float(states[flip_j]["objective"] - exact_energy), 0.0)
    delta_both = max(float(states[flip_both]["objective"] - exact_energy), 0.0)
    # G_ij mede quão rigidamente o melhor estado condicionado do par se separa do segundo.
    gij = max(float(states[second_state]["objective"] - states[best_state]["objective"]), 0.0)
    pair_rows.append({
        "asset_index_i": i,
        "asset_index_j": j,
        "asset_i": tickers[i],
        "asset_j": tickers[j],
        "pair": f"{tickers[i]}/{tickers[j]}",
        "global_pair_state": global_state,
        "G_ij": gij,
        "G_ij_relative": gij / max(energy_span, ENERGY_ATOL),
        "single_flip_i_gap": delta_i,
        "single_flip_j_gap": delta_j,
        "joint_flip_gap": delta_both,
        "joint_flip_nonadditivity": delta_both - delta_i - delta_j,
        **{f"E_{state}": float(states[state]["objective"]) for state in PAIR_STATES},
    })

pair_gap_df = pd.DataFrame(pair_rows).sort_values("G_ij", ascending=False).reset_index(drop=True)

print("Portfólios válidos:", len(enumeration_df))
print("Energia exata:", exact_energy)
print("Bitstring ótimo — ordem dos ativos:", exact_asset_bitstrings)
print("Bitstring ótimo — ordem Qiskit:", exact_qiskit_bitstrings)
print("Ativos selecionados:", optimal_df.iloc[0]["selected_assets"])
display(asset_decision_df)
display(pair_gap_df.head(15))


# Parte III — reconstrução exata do Hamiltoniano e do ansatz do modelo 20.1

A construção abaixo foi separada do gerador de banco, mas preserva a mesma lógica do notebook 20.1:

- mesmo QUBO/Ising;
- mesmo estado inicial de peso $k$;
- mesmos blocos `CY` e `CCY`;
- mesma ordenação rastreável dos parâmetros;
- mesmo `ANSATZ_SEED`.

O notebook interrompe a execução caso o vetor salvo tenha dimensão incompatível ou caso a auditoria estrutural falhe.


### Célula 6 — Dependências quânticas sem importação de otimizadores

**Em termos simples:** esta célula carrega somente as classes necessárias para montar o QUBO, converter para Ising, construir o circuito e calcular o `Statevector`.

Nenhum método como COBYLA, SPSA ou outro otimizador variacional é importado. Isso garante que as células seguintes apenas atribuam valores de $\theta$ e avaliem diretamente o circuito.


In [ ]:
# ============================================================
# 6. DEPENDÊNCIAS QUÂNTICAS — SEM OTIMIZADOR
# ============================================================

# Dependências para formular o problema binário e construir o circuito.
# Não há importação de classes de otimização variacional.
from docplex.mp.model import Model
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

print("Dependências quânticas carregadas. Nenhum otimizador foi importado.")


### Célula 7 — Construção do QUBO e do Hamiltoniano de Ising

**Em termos simples:** o problema financeiro clássico é escrito com variáveis binárias $x_i\in\{0,1\}$ e a restrição de selecionar exatamente $k$ ativos:

$$
\sum_i x_i=k.
$$

O modelo é convertido para QUBO e depois para um Hamiltoniano de Ising:

$$
H=\sum_i h_i Z_i+\sum_{i<j}J_{ij}Z_iZ_j+\text{constante}.
$$

A célula exige que o Hamiltoniano seja diagonal, isto é, sem termos `X` ou `Y`. Em seguida, compara a energia mínima do Ising com a energia obtida pela enumeração dos 210 portfólios. Se elas não coincidirem, a execução para.

**Saídas principais:** `ising`, `ising_offset` e `hamiltonian_terms_df`.


In [ ]:
# ============================================================
# 7. QUBO E HAMILTONIANO DE ISING
# ============================================================


def build_docplex_ising():
    """Constrói o modelo binário, converte para QUBO/Ising e audita a energia exata."""
    model = Model(name=f"portfolio_n{N_ASSETS}_k{TARGET_K}")
    variables = np.array(
        [model.binary_var(name=f"x_{index}") for index in range(N_ASSETS)],
        dtype=object,
    )
    # Termo quadrático de risco x^T Sigma x.
    risk_expression = model.sum(
        float(sigma[row, column]) * variables[row] * variables[column]
        for row in range(N_ASSETS)
        for column in range(N_ASSETS)
    )
    # Termo linear de retorno esperado mu^T x.
    return_expression = model.sum(
        float(mu[index]) * variables[index]
        for index in range(N_ASSETS)
    )
    model.minimize(
        Q_VALUE * risk_expression
        - (1.0 - Q_VALUE) * return_expression
        + RISK_FREE
    )
    # Restrição de cardinalidade: exatamente TARGET_K variáveis devem valer 1.
    model.add_constraint(
        model.sum(variables.tolist()) == int(TARGET_K),
        ctname="budget",
    )

    # Docplex -> QuadraticProgram -> QUBO -> operador Ising e deslocamento constante.
    quadratic_program = from_docplex_mp(model=model)
    qubo = QuadraticProgramToQubo().convert(quadratic_program)
    ising, offset = qubo.to_ising()

    # O problema de portfólio deve gerar apenas termos diagonais I, Z e ZZ.
    labels = [str(label) for label in ising.paulis.to_labels()]
    non_diagonal = [label for label in labels if "X" in label or "Y" in label]
    if non_diagonal:
        raise RuntimeError(f"Hamiltoniano não diagonal: {non_diagonal[:10]}")

    # Auditoria independente: a menor energia Ising deve coincidir com a enumeração.
    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(operator=ising)
    exact_energy_ising = float(np.real(exact_result.eigenvalue + offset))
    if not np.isclose(exact_energy_ising, exact_energy, atol=1e-10, rtol=0.0):
        raise RuntimeError(
            "Energia Ising e enumeração clássica não coincidem: "
            f"{exact_energy_ising} vs {exact_energy}"
        )

    terms_df = pd.DataFrame({
        "pauli_label": labels,
        "coefficient": np.real(np.asarray(ising.coeffs)).astype(float),
        "body_order": [label.count("Z") for label in labels],
    })
    return model, quadratic_program, qubo, ising, float(offset), terms_df


model, quadratic_program, qubo, ising, ising_offset, hamiltonian_terms_df = build_docplex_ising()
print("Termos Ising:", len(hamiltonian_terms_df))
display(hamiltonian_terms_df.sort_values(["body_order", "pauli_label"]))


### Célula 8 — Construção rastreável do ansatz de Dicke

**Em termos simples:** esta célula reproduz o circuito parametrizado usado no experimento 20.1 e registra a origem estrutural de cada parâmetro.

- `CY_parameterized` cria um bloco lógico de dois qubits cuja operação parametrizada primitiva é `CRY`;
- `CCY_parameterized` cria um bloco lógico de três qubits usando `RY` e `CCX`;
- as portas `X` iniciais preparam apenas um estado-base com peso de Hamming $k$;
- cada parâmetro recebe informações como posição, distância entre qubits e tipo de bloco.

O número esperado de parâmetros é

$$
N_\theta=\frac{k(2n-k-1)}{2}.
$$

**Saídas principais:** `ansatz`, `structure_df`, `initial_x_qubits` e `N_PARAMETERS`.


In [ ]:
# ============================================================
# 8. PORTAS E ANSATZ DE DICKE RASTREÁVEL — CÓPIA DA CABEÇA 20.1
# ============================================================


def CY_parameterized(identifier):
    """Cria o bloco lógico CY com uma rotação controlada CRY(theta)."""
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")


def CCY_parameterized(identifier):
    """Cria o bloco lógico CCY com rotações RY(theta) e controles CCX."""
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")


def dicke_parameter_count(n_value, k_value):
    """Número esperado de parâmetros do ansatz: k(2n-k-1)/2."""
    return int(k_value * (2 * n_value - k_value - 1) / 2)


def build_tracked_dicke_ansatz(n_value, k_value, seed):
    """Constrói o ansatz e registra a origem lógica de cada parâmetro."""
    # Preserva o estado aleatório global para que a construção do ansatz não
    # altere outras rotinas aleatórias do notebook.
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)

        # Prepara um estado-base com exatamente k excitações; não injeta o ótimo clássico.
        initial_x_qubits = []
        for excitation_index in range(k_value):
            qubit = n_value - excitation_index - 1
            qc.x(qubit)
            initial_x_qubits.append(qubit)

        # Cada bloco criado gera um registro que depois será ligado ao theta_index real.
        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"
                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    # A ordem dos parâmetros é extraída do circuito decomposto, a mesma forma usada
    # nas avaliações por Statevector.
    decomposed = qc.decompose()
    ordered_parameters = list(decomposed.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(ordered_parameters)
    }
    structure_rows = []
    for record in records:
        record = record.copy()
        parameter_object = record.pop("parameter_object")
        structure_rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **record,
        })

    structure_df = pd.DataFrame(structure_rows).sort_values("theta_index").reset_index(drop=True)
    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(f"Esperados {expected} parâmetros; encontrados {len(structure_df)}.")

    structure_df["n"] = int(n_value)
    structure_df["k"] = int(k_value)
    structure_df["rho_k_over_n"] = k_value / n_value
    structure_df["u_l"] = structure_df["l"] / max(n_value - 1, 1)
    structure_df["u_distance"] = structure_df["distance"] / max(k_value, 1)
    structure_df["u_theta_index"] = structure_df["theta_index"] / max(expected - 1, 1)
    return decomposed, structure_df, tuple(sorted(initial_x_qubits))


ANSATZ_SEED = RANDOM_SEED + 100 * N_ASSETS + TARGET_K
ansatz, structure_df, initial_x_qubits = build_tracked_dicke_ansatz(
    N_ASSETS, TARGET_K, ANSATZ_SEED
)
N_PARAMETERS = int(ansatz.num_parameters)

print("ANSATZ_SEED =", ANSATZ_SEED)
print("n_parameters =", N_PARAMETERS)
print("qubits com X inicial =", initial_x_qubits)
print("estado-base inicial em ordem Qiskit =", "".join(
    "1" if qubit in initial_x_qubits else "0"
    for qubit in range(N_ASSETS - 1, -1, -1)
))


### Célula 9 — Mapa físico, lógico e financeiro de cada $\theta_j$

**Em termos simples:** esta célula percorre o circuito decomposto e identifica em qual operação física cada parâmetro aparece, quais qubits ele toca e em que posições do circuito ele é usado.

A periodicidade **estrutural da unidade** é definida por:

$$
T_j=
\begin{cases}
4\pi, & \text{se o parâmetro aparece em uma porta CRY},\\
2\pi, & \text{se aparece somente em RY}.
\end{cases}
$$

Uma `RY` muda apenas por uma fase global após $2\pi$. Em uma `CRY`, essa troca de sinal ocorre somente no setor em que o controle vale 1 e pode se tornar uma fase relativa observável; por isso o período seguro da unidade controlada é $4\pi$.

Isso explica por que `theta_2` e `theta_3`, que pertencem a blocos `CY/CRY`, são varridos de zero a $4\pi$. Os demais parâmetros mostrados pertencem a blocos `CCY/RY` e são varridos de zero a $2\pi$.

Entretanto, o observável específico $P(\mathcal{X}_{\mathrm{opt}})$ pode repetir após $2\pi$ mesmo quando a unidade tem período $4\pi$. Por isso a versão 20.6 compara numericamente a primeira e a segunda metade das curvas `CRY` nos 100 vetores.

Os **ativos não mudam durante uma varredura**. Cada $\theta_j$ está ligado a um bloco lógico fixo do ansatz e, portanto, a qubits/ativos fixos. Os nomes são diferentes entre `theta_17`, `theta_25` etc. porque são parâmetros de blocos diferentes, não porque o código esteja trocando ações entre os vetores.

**Saídas principais:** `parameter_map_df`, `parameter_occurrence_df` e `structural_audit`.

In [ ]:
# ============================================================
# 9. MAPA FÍSICO, LÓGICO E FINANCEIRO DOS PARÂMETROS
# ============================================================


def build_physical_parameter_map(ansatz, structure_df):
    """Liga cada theta a operações primitivas, qubits, período e posição no circuito."""
    parameter_order = list(ansatz.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(parameter_order)
    }
    occurrence_rows = []

    # Percorre cada instrução do circuito decomposto e registra todas as ocorrências
    # de parâmetros nas expressões angulares das portas.
    for instruction_position, instruction in enumerate(ansatz.data):
        operation = instruction.operation
        operation_name = str(operation.name).lower()
        qubits = tuple(
            int(ansatz.find_bit(qubit).index)
            for qubit in instruction.qubits
        )
        for parameter_slot, expression in enumerate(operation.params):
            for parameter in getattr(expression, "parameters", set()):
                if parameter not in parameter_to_index:
                    continue
                try:
                    coefficient = float(expression.gradient(parameter))
                except Exception:
                    coefficient = np.nan
                occurrence_rows.append({
                    "theta_index": int(parameter_to_index[parameter]),
                    "parameter_name": str(parameter),
                    "primitive_operation": operation_name,
                    "primitive_qubits": qubits,
                    "instruction_position": int(instruction_position),
                    "parameter_slot": int(parameter_slot),
                    "parameter_coefficient": coefficient,
                })

    # Uma linha por ocorrência física de parâmetro.
    occurrence_df = pd.DataFrame(occurrence_rows)
    physical_rows = []
    for theta_index, group in occurrence_df.groupby("theta_index"):
        operations = sorted(set(group["primitive_operation"].astype(str)))
        primitive_qubits = tuple(sorted({
            int(qubit)
            for qubit_tuple in group["primitive_qubits"]
            for qubit in qubit_tuple
        }))
        # RY(theta+2pi) difere apenas por fase global, mas CRY pode exigir 4pi porque
        # o sinal relativo entre os setores do qubit de controle é observável.
        if "cry" in operations:
            primitive_type = "CRY"
            angular_period = float(4 * np.pi)
            periodicity_class = "four_pi_eligible"
        else:
            primitive_type = "RY"
            angular_period = float(2 * np.pi)
            periodicity_class = "guaranteed_2pi"
        physical_rows.append({
            "theta_index": int(theta_index),
            "primitive_physical_type": primitive_type,
            "primitive_operations": tuple(operations),
            "primitive_parameter_qubits": primitive_qubits,
            "angular_period": angular_period,
            "periodicity_structural_class": periodicity_class,
            "n_occurrences_decomposed": int(len(group)),
            "first_instruction": int(group["instruction_position"].min()),
            "last_instruction": int(group["instruction_position"].max()),
        })

    # Une a descrição lógica do bloco à descrição física observada após decomposição.
    parameter_map = structure_df.merge(
        pd.DataFrame(physical_rows),
        on="theta_index",
        how="left",
        validate="one_to_one",
    ).sort_values("theta_index").reset_index(drop=True)

    def logical_qubits(row):
        """Retorna todos os qubits pertencentes ao bloco lógico CY ou CCY."""
        i_value, l_value = int(row["i"]), int(row["l"])
        if row["ansatz_gate_type"] == "CY":
            return tuple(sorted((i_value, l_value)))
        return tuple(sorted((i_value, i_value + 1, l_value)))

    parameter_map["logical_block_qubits"] = parameter_map.apply(logical_qubits, axis=1)
    parameter_map["logical_assets"] = parameter_map["logical_block_qubits"].map(
        lambda qubits: tuple(tickers[index] for index in qubits)
    )
    parameter_map["block_size"] = parameter_map["logical_block_qubits"].map(len)

    # Auditoria estrutural: qualquer inconsistência interrompe o experimento.
    checks = {
        "all_parameters_mapped": bool(len(parameter_map) == N_PARAMETERS),
        "CY_is_CRY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "primitive_physical_type",
        ].eq("CRY").all()),
        "CCY_is_RY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "primitive_physical_type",
        ].eq("RY").all()),
        "CY_has_two_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "block_size",
        ].eq(2).all()),
        "CCY_has_three_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "block_size",
        ].eq(3).all()),
    }
    failed = [name for name, value in checks.items() if not value]
    if failed:
        raise RuntimeError(f"Auditoria estrutural falhou: {failed}")

    return parameter_map, occurrence_df, checks


parameter_map_df, parameter_occurrence_df, structural_audit = build_physical_parameter_map(
    ansatz, structure_df
)

# Liga cada bloco aos pares clássicos contidos nos seus qubits lógicos.
def pair_metrics_for_block(qubits):
    """Resume os gaps clássicos dos pares de ativos internos ao bloco."""
    pairs = {tuple(sorted(pair)) for pair in combinations(qubits, 2)}
    subset = pair_gap_df.loc[
        pair_gap_df.apply(
            lambda row: tuple(sorted((int(row["asset_index_i"]), int(row["asset_index_j"])))) in pairs,
            axis=1,
        )
    ]
    return pd.Series({
        "n_internal_asset_pairs": int(len(subset)),
        "max_internal_G_ij": float(subset["G_ij"].max()),
        "mean_internal_G_ij": float(subset["G_ij"].mean()),
        "max_internal_joint_gap": float(subset["joint_flip_gap"].max()),
        "max_internal_nonadditivity_abs": float(subset["joint_flip_nonadditivity"].abs().max()),
        "internal_pairs": tuple(subset["pair"].tolist()),
    })

block_pair_metrics = parameter_map_df["logical_block_qubits"].apply(pair_metrics_for_block)
parameter_map_df = pd.concat([parameter_map_df, block_pair_metrics], axis=1)

print(json.dumps(structural_audit, indent=2, ensure_ascii=False))
display(parameter_map_df)


## Auditoria visual da criação do circuito

A tabela anterior é a ponte entre o índice local e a descrição transferível:

- `theta_index`: posição local neste circuito;
- `ansatz_gate_type`: bloco lógico `CY` ou `CCY`;
- `primitive_physical_type`: operação parametrizada observada após decomposição;
- `logical_block_qubits`: qubits realmente envolvidos pelo bloco;
- `logical_assets`: ativos associados a esses qubits;
- `max_internal_G_ij`: maior rigidez clássica entre os pares internos do bloco;
- `angular_period`: domínio máximo usado na varredura.

A célula seguinte mostra o circuito e contabiliza as operações. O estado preparado pelas portas `X` é apenas a semente de peso $k$, não o bitstring ótimo clássico.


### Célula 10 — Auditoria visual do circuito

**Em termos simples:** esta célula mostra quantas operações de cada tipo existem, exibe a tabela estrutural dos parâmetros e tenta desenhar o ansatz.

A tabela permite conferir, para cada `theta_index`, o bloco lógico, a operação física, os qubits/ativos tocados, o período angular e a rigidez clássica interna.

Caso o desenho com Matplotlib não esteja disponível, o circuito é mostrado em formato de texto.


In [ ]:
# ============================================================
# 10. VISUALIZAÇÃO E CONTAGEM DAS PORTAS
# ============================================================

# Contagem de portas no circuito já decomposto.
operation_counts = pd.DataFrame(
    sorted(ansatz.count_ops().items()),
    columns=["operation", "count"],
)

display(operation_counts)
display(parameter_map_df[[
    "theta_index",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "distance",
    "first_instruction",
    "last_instruction",
    "angular_period",
    "internal_pairs",
    "max_internal_G_ij",
]])

# Tenta o desenho gráfico; o modo texto é um fallback para ambientes sem suporte.
try:
    display(ansatz.draw(output="mpl", fold=120))
except Exception as exc:
    warnings.warn(f"Desenho matplotlib não disponível: {type(exc).__name__}: {exc}")
    print(ansatz.draw(output="text", fold=120))


# Parte IV — compatibilidade entre `merge.pkl` e o circuito reconstruído

Antes de interpretar qualquer varredura, o notebook verifica:

1. todos os vetores possuem a dimensão esperada;
2. vetores salvos podem ser atribuídos ao circuito;
3. a distribuição exata recalculada é compatível com as colunas salvas;
4. nenhuma avaliação chama otimizador.


### Célula 11 — Avaliação exata de um vetor $\theta$

**Em termos simples:** esta é a função central usada em todas as intervenções posteriores. Ela recebe um vetor $\theta$, atribui os valores ao ansatz e calcula diretamente o estado quântico:

$$
|\psi(\theta)\rangle=U(\theta)|\psi_0\rangle.
$$

A função `evaluate_theta` devolve, entre outras métricas:

- probabilidade total dos bitstrings ótimos;
- energia esperada $\langle H\rangle$ e gap para a energia exata;
- bitstring dominante;
- massa dentro do subespaço válido de cardinalidade $k$;
- entropia e razão de participação;
- massa acumulada nos 1, 5 e 10 melhores portfólios clássicos.

Quando existe uma distribuição de referência, também são calculadas a distância de variação total

$$
\operatorname{TVD}(p,q)=\frac{1}{2}\sum_z|p_z-q_z|
$$

e a divergência de Jensen–Shannon.

Ao final, uma pequena amostra do banco é reavaliada para verificar compatibilidade entre os vetores salvos e o circuito reconstruído.


In [ ]:
# ============================================================
# 11. FUNÇÕES DE PARSE DOS VETORES E MÉTRICAS EXATAS
# ============================================================

# Converte todos os vetores salvos e mantém apenas aqueles compatíveis com o
# número de parâmetros do ansatz reconstruído.
best_parameters_column = RESOLVED_COLUMNS["best_parameters"]
merge_work_df = merge_df.copy()
merge_work_df["theta_vector"] = merge_work_df[best_parameters_column].map(parse_vector)
merge_work_df["theta_dimension"] = merge_work_df["theta_vector"].map(len)

dimension_counts = merge_work_df["theta_dimension"].value_counts().sort_index()
print("Dimensões encontradas:")
display(dimension_counts.rename_axis("theta_dimension").to_frame("rows"))

merge_work_df = merge_work_df.loc[
    merge_work_df["theta_dimension"].eq(N_PARAMETERS)
].copy()
if merge_work_df.empty:
    raise RuntimeError(
        f"Nenhum vetor do merge possui a dimensão esperada de {N_PARAMETERS} parâmetros."
    )

# Mapeamento entre índices do Statevector e bitstrings na convenção do Qiskit.
all_labels = np.asarray(
    [format(index, f"0{N_ASSETS}b") for index in range(2 ** N_ASSETS)],
    dtype=object,
)
label_to_index = {str(label): int(index) for index, label in enumerate(all_labels)}
valid_bitstrings = enumeration_df["bitstring_qiskit_order"].astype(str).to_numpy(dtype=object)
valid_indices = np.asarray([label_to_index[bitstring] for bitstring in valid_bitstrings], dtype=int)
valid_objectives = enumeration_df["objective"].to_numpy(dtype=float)
optimal_indices = np.asarray([label_to_index[bitstring] for bitstring in exact_qiskit_bitstrings], dtype=int)
optimal_set = set(exact_qiskit_bitstrings)


def total_variation_distance(p, q):
    """Calcula TVD(p,q)=0.5*sum(|p-q|), entre 0 e 1."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(0.5 * np.sum(np.abs(p - q)))


def jensen_shannon_divergence(p, q, epsilon=1e-15):
    """Calcula uma divergência simétrica e finita entre duas distribuições."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(p.sum(), epsilon)
    q = q / max(q.sum(), epsilon)
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + epsilon) / (m + epsilon)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + epsilon) / (m + epsilon)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


def circular_distance(a, b, period):
    """Menor distância entre dois ângulos em um círculo de período conhecido."""
    return float(abs((float(a) - float(b) + 0.5 * period) % period - 0.5 * period))


def shortest_delta_to_target(source, target, period):
    """Deslocamento assinado mais curto da origem até o alvo periódico."""
    return float((float(target) - float(source) + 0.5 * period) % period - 0.5 * period)


def statevector_from_theta(theta):
    """Retorna o vetor de estado complexo após atribuição direta dos parâmetros."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")
    assigned = ansatz.assign_parameters(theta, inplace=False)
    return np.asarray(Statevector.from_instruction(assigned).data, dtype=np.complex128)


def evaluate_theta(
    theta,
    reference_probability=None,
    return_valid_probability=False,
    return_statevector=False,
):
    """Atribui theta ao ansatz e calcula exatamente estado, energia e probabilidades."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")

    # Intervenção direta: não existe passo de otimização entre atribuir theta e avaliar.
    state_data = statevector_from_theta(theta)
    state = Statevector(state_data)
    probability = np.asarray(state.probabilities(), dtype=float)
    # Restringe a distribuição aos C(n,k) bitstrings que respeitam a cardinalidade.
    valid_probability = probability[valid_indices]
    valid_mass = float(valid_probability.sum())
    leakage = max(0.0, 1.0 - valid_mass)
    p_optimal = float(probability[optimal_indices].sum())
    # Energia física completa = valor esperado do operador + offset da conversão QUBO.
    energy = float(np.real(state.expectation_value(ising)) + ising_offset)
    dominant_index = int(np.argmax(probability))
    dominant_bitstring = str(all_labels[dominant_index])

    nonzero = valid_probability[valid_probability > 0.0]
    entropy = float(-np.sum(nonzero * np.log(nonzero)))
    normalized_entropy = float(entropy / np.log(len(valid_probability)))
    participation = float(1.0 / np.sum(valid_probability ** 2))

    dominant_valid_position = int(np.argmax(valid_probability))
    dominant_valid_rank = dominant_valid_position + 1

    result = {
        "p_optimal": p_optimal,
        "expected_energy": energy,
        "energy_gap": float(abs(energy - exact_energy)),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probability[dominant_index]),
        "dominant_is_optimal": bool(dominant_bitstring in optimal_set),
        "dominant_valid_rank": dominant_valid_rank,
        "p_top_1_classical": float(valid_probability[:1].sum()),
        "p_top_5_classical": float(valid_probability[:5].sum()),
        "p_top_10_classical": float(valid_probability[:10].sum()),
        "valid_dicke_mass": valid_mass,
        "leakage_outside_k": leakage,
        "normalized_entropy_valid": normalized_entropy,
        "participation_ratio_valid": participation,
    }
    if reference_probability is not None:
        result["tvd_vs_anchor"] = total_variation_distance(probability, reference_probability)
        result["jsd_vs_anchor"] = jensen_shannon_divergence(probability, reference_probability)
    if return_valid_probability:
        result["valid_probability"] = valid_probability.astype(np.float32)
    if return_statevector:
        result["statevector"] = state_data
    result["full_probability"] = probability
    return result


# Auditoria em uma amostra pequena antes das campanhas longas.
# Compara probabilidade e bitstring salvos com a reavaliação exata atual.
audit_indices = merge_work_df.sample(min(10, len(merge_work_df)), random_state=RANDOM_SEED).index
compatibility_rows = []
for row_index in audit_indices:
    row = merge_work_df.loc[row_index]
    metrics = evaluate_theta(row["theta_vector"])
    saved_p = (
        pd.to_numeric(pd.Series([row[RESOLVED_COLUMNS["p_best"]]]), errors="coerce").iloc[0]
        if RESOLVED_COLUMNS["p_best"] is not None
        else np.nan
    )
    saved_bit = (
        normalize_bitstring(row[RESOLVED_COLUMNS["dominant_bitstring"]])
        if RESOLVED_COLUMNS["dominant_bitstring"] is not None
        else None
    )
    compatibility_rows.append({
        "row_index": row_index,
        "saved_p": saved_p,
        "exact_p_recomputed": metrics["p_optimal"],
        "p_difference": metrics["p_optimal"] - saved_p if np.isfinite(saved_p) else np.nan,
        "saved_dominant": saved_bit,
        "exact_dominant_recomputed": metrics["dominant_bitstring"],
        "dominant_equal": bool(saved_bit == metrics["dominant_bitstring"]) if saved_bit else np.nan,
        "exact_energy_recomputed": metrics["expected_energy"],
        "leakage": metrics["leakage_outside_k"],
    })

compatibility_audit_df = pd.DataFrame(compatibility_rows)
display(compatibility_audit_df)

if compatibility_audit_df["leakage"].max() > 1e-10:
    raise RuntimeError("O circuito reconstruído apresentou vazamento para fora do subespaço k=4.")


### Célula 12 — Máscara dos 10% melhores e seleção de 100 vetores

**Em termos simples:** esta célula reduz o banco aos vetores mais promissores e escolhe 100 vetores completos para a campanha de varredura.

Para cada linha válida são construídos dois rankings percentuais:

- $R_p$: cresce quando a probabilidade salva da solução ótima aumenta;
- $R_{\mathrm{gap}}$: cresce quando o gap de energia diminui.

O score salvo é

$$
Q_{\mathrm{salvo}}=\frac{R_p+R_{\mathrm{gap}}}{2}.
$$

A máscara mantém aproximadamente os 10% superiores. Até 300 candidatos são reavaliados exatamente com `evaluate_theta`, e os 100 melhores pelo score exato são selecionados.

**Importante:** cada âncora é uma linha inteira do banco. A máscara não mistura componentes de vetores diferentes, não zera componentes internos e não muda os ativos associados a cada índice $\theta_j$.

In [ ]:
# ============================================================
# 12. MÁSCARA DOS 10% MELHORES E SELEÇÃO DE 100 VETORES
# ============================================================

# Recupera os nomes reais das colunas que alimentarão a seleção.
objective_col = RESOLVED_COLUMNS["objective"]
best_objective_col = RESOLVED_COLUMNS["best_objective"]
p_col = RESOLVED_COLUMNS["p_best"]
gap_col = RESOLVED_COLUMNS["gap"]
status_col = RESOLVED_COLUMNS["status"]

bank = merge_work_df.copy()
bank["saved_p"] = (
    pd.to_numeric(bank[p_col], errors="coerce") if p_col is not None else np.nan
)
bank["saved_objective"] = (
    pd.to_numeric(bank[objective_col], errors="coerce") if objective_col is not None else np.nan
)

# Prioridade para o gap:
# 1) usa a coluna pronta; 2) calcula objetivo - melhor objetivo;
# 3) compara a energia salva com a referência clássica exata.
if gap_col is not None:
    bank["saved_gap"] = pd.to_numeric(bank[gap_col], errors="coerce").abs()
elif best_objective_col is not None and objective_col is not None:
    bank["saved_gap"] = (
        pd.to_numeric(bank[objective_col], errors="coerce")
        - pd.to_numeric(bank[best_objective_col], errors="coerce")
    ).abs()
else:
    bank["saved_gap"] = (bank["saved_objective"] - exact_energy).abs()

if status_col is not None:
    status_mask = bank[status_col].astype(str).str.lower().eq("ok")
else:
    status_mask = pd.Series(True, index=bank.index)

valid_bank = bank.loc[
    status_mask
    & bank["theta_vector"].notna()
    & bank["saved_gap"].notna()
].copy()

if valid_bank.empty:
    raise RuntimeError("Nenhum vetor válido foi encontrado para a seleção das âncoras.")

if valid_bank["saved_p"].notna().any():
    valid_bank["p_quality_rank"] = valid_bank["saved_p"].rank(
        pct=True, ascending=True, method="average"
    )
else:
    valid_bank["p_quality_rank"] = 0.5

valid_bank["gap_quality_rank"] = valid_bank["saved_gap"].rank(
    pct=True, ascending=False, method="average"
)
valid_bank["quality_score_saved"] = 0.5 * (
    valid_bank["p_quality_rank"] + valid_bank["gap_quality_rank"]
)

threshold = float(valid_bank["quality_score_saved"].quantile(1.0 - TOP_FRACTION))
top_masked_bank = valid_bank.loc[
    valid_bank["quality_score_saved"].ge(threshold)
].copy()

if len(top_masked_bank) < N_ANCHORS:
    raise RuntimeError(
        f"A máscara produziu somente {len(top_masked_bank)} vetores, mas a campanha "
        f"exige {N_ANCHORS}. Aumente TOP_FRACTION ou verifique o banco."
    )

candidate_count = min(
    max(int(MAX_EXACT_REEVALUATION), int(N_ANCHORS)),
    len(top_masked_bank),
)
candidate_pool = top_masked_bank.sort_values(
    ["quality_score_saved", "saved_p", "saved_gap"],
    ascending=[False, False, True],
).head(candidate_count)

# Reavalia todos os candidatos sob o mesmo circuito e a mesma referência.
exact_candidate_rows = []
for row_index, row in candidate_pool.iterrows():
    row_hash = problem_fingerprint(row)
    if row_hash != DATA_HASH:
        raise RuntimeError(
            f"A linha {row_index} pertence a outro Hamiltoniano: {row_hash} != {DATA_HASH}."
        )
    metrics = evaluate_theta(row["theta_vector"])
    exact_candidate_rows.append({
        "source_row_index": row_index,
        "problem_hash": row_hash,
        "theta_vector": row["theta_vector"],
        "saved_p": row["saved_p"],
        "saved_gap": row["saved_gap"],
        "quality_score_saved": row["quality_score_saved"],
        **{key: value for key, value in metrics.items() if key != "full_probability"},
    })

exact_candidates_df = pd.DataFrame(exact_candidate_rows)
exact_candidates_df["p_rank_exact"] = exact_candidates_df["p_optimal"].rank(
    pct=True, ascending=True
)
exact_candidates_df["gap_rank_exact"] = exact_candidates_df["energy_gap"].rank(
    pct=True, ascending=False
)
exact_candidates_df["quality_score_exact"] = 0.5 * (
    exact_candidates_df["p_rank_exact"] + exact_candidates_df["gap_rank_exact"]
)

anchors_df = exact_candidates_df.sort_values(
    ["quality_score_exact", "p_optimal", "energy_gap"],
    ascending=[False, False, True],
).head(N_ANCHORS).reset_index(drop=True)
anchors_df.insert(0, "anchor_id", np.arange(len(anchors_df), dtype=int))

if len(anchors_df) != N_ANCHORS:
    raise RuntimeError(
        f"Esperados {N_ANCHORS} vetores finais; encontrados {len(anchors_df)}."
    )
if anchors_df["problem_hash"].nunique() != 1 or anchors_df["problem_hash"].iloc[0] != DATA_HASH:
    raise RuntimeError("Os 100 vetores selecionados não pertencem ao mesmo Hamiltoniano.")

print("linhas válidas no banco:", len(valid_bank))
print("linhas após máscara superior:", len(top_masked_bank))
print("candidatos reavaliados exatamente:", len(exact_candidates_df))
print("vetores finais selecionados:", len(anchors_df))
print("Hamiltoniano único:", anchors_df["problem_hash"].unique().tolist())
display(anchors_df.drop(columns=["theta_vector"], errors="ignore"))

# Parte V — varreduras individuais em 100 vetores

Cada um dos 100 vetores selecionados pela máscara é usado como uma âncora independente. Para cada parâmetro testado, somente um componente é alterado; os outros 29 permanecem exatamente iguais aos valores daquele mesmo vetor.

Parâmetros detalhados:

$$
\{17,2,14,19,22,25,27,3,24\}.
$$

Cada parâmetro é varrido no período estrutural da porta. O valor original canônico do componente é inserido explicitamente na grade, mesmo quando não coincide com os 201 pontos regulares.

Não há média entre vetores na geração das curvas. A média, a mediana e os quantis aparecem apenas depois, como resumos transparentes da coleção de 100 curvas brutas.

### Célula 13 — Varredura individual de um parâmetro por vez em cada vetor

Para cada vetor $\theta^{(a)}$, com $a\in\{0,\ldots,99\}$, a célula avalia

$$
\theta^{(a,j)}(\phi)
=
\bigl(
\theta_0^{(a)},\ldots,\theta_{j-1}^{(a)},
\phi,
\theta_{j+1}^{(a)},\ldots
\bigr),
\qquad 0\leq\phi\leq T_j.
$$

A auditoria causal confirma, em cada avaliação, que o único índice alterado é exatamente `theta_index`. O ponto original é inserido uma única vez e precisa reproduzir a probabilidade da âncora correspondente.

Cada tarefa `(anchor_id, theta_index)` possui checkpoint próprio. Assim, uma execução interrompida pode continuar sem refazer as varreduras já concluídas.

In [ ]:
# ============================================================
# 13. MOTOR DE VARREDURA INDIVIDUAL — 100 VETORES, SEM COBYLA
# ============================================================


def inclusive_grid(start, end, step=None, n_points=None):
    """Cria uma grade que inclui explicitamente os limites start e end."""
    start, end = float(start), float(end)
    if step is not None:
        values = np.arange(start, end, float(step), dtype=float)
        if len(values) == 0 or not np.isclose(values[-1], end, atol=1e-12, rtol=0.0):
            values = np.append(values, end)
        else:
            values[-1] = end
        return values
    if n_points is None or n_points < 2:
        raise ValueError("Informe step ou n_points >= 2.")
    return np.linspace(start, end, int(n_points), endpoint=True)


def parameter_period(theta_index):
    """Recupera em parameter_map_df o período estrutural 2pi ou 4pi."""
    row = parameter_map_df.loc[
        parameter_map_df["theta_index"].eq(int(theta_index))
    ]
    if len(row) != 1:
        raise KeyError(f"theta_{theta_index} não foi identificado de forma única.")
    return float(row.iloc[0]["angular_period"])


def run_single_parameter_task(
    anchor_row,
    theta_index,
    n_points=SWEEP_POINTS_PER_PERIOD,
):
    """Varre um theta de uma âncora, mantendo todos os demais componentes fixos."""
    anchor_id = int(anchor_row["anchor_id"])
    theta_index = int(theta_index)
    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float).copy()

    period = parameter_period(theta_index)
    original_raw = float(theta_anchor[theta_index])
    original_canonical = float(original_raw % period)

    # Grade regular por período + valor original inserido exatamente.
    base_grid = inclusive_grid(0.0, period, n_points=int(n_points))
    grid = np.unique(np.append(base_grid, original_canonical))
    grid.sort()
    original_grid_index = int(np.argmin(np.abs(grid - original_canonical)))
    if not np.isclose(
        grid[original_grid_index], original_canonical, atol=1e-14, rtol=0.0
    ):
        raise RuntimeError(f"Não foi possível inserir o ponto original de theta_{theta_index}.")

    task_stem = f"anchor_{anchor_id:03d}_theta_{theta_index:02d}"
    summary_path = CHECKPOINT_DIR / f"detailed_{task_stem}.pkl"
    distribution_path = CHECKPOINT_DIR / f"detailed_{task_stem}_valid_probabilities.npz"

    if REUSE_SWEEP_CHECKPOINTS and summary_path.exists() and distribution_path.exists():
        cached = pd.read_pickle(summary_path)
        required = {
            "anchor_id", "theta_index", "is_original_theta", "parameter_name",
            "theta_value", "p_optimal", "period",
        }
        if required.issubset(cached.columns):
            return cached, distribution_path

    anchor_metrics = evaluate_theta(theta_anchor)
    reference_probability = anchor_metrics["full_probability"]
    parameter_name = str(list(ansatz.parameters)[theta_index])

    rows = []
    probability_matrix = np.empty((len(grid), len(valid_indices)), dtype=np.float32)

    for grid_index, value in enumerate(grid):
        theta_test = theta_anchor.copy()
        theta_test[theta_index] = float(value)

        changed_indices = np.flatnonzero(
            ~np.isclose(theta_test, theta_anchor, atol=1e-14, rtol=0.0)
        )
        is_original = bool(grid_index == original_grid_index)
        if is_original:
            if len(changed_indices) != 0 and not (
                len(changed_indices) == 1 and int(changed_indices[0]) == theta_index
            ):
                raise RuntimeError(
                    f"Auditoria falhou no ponto original de theta_{theta_index}, "
                    f"âncora {anchor_id}: índices alterados = {changed_indices.tolist()}"
                )
        elif changed_indices.tolist() != [theta_index]:
            raise RuntimeError(
                f"Auditoria falhou em theta_{theta_index}, âncora {anchor_id}: "
                f"índices alterados = {changed_indices.tolist()}"
            )

        metrics = evaluate_theta(
            theta_test,
            reference_probability=reference_probability,
            return_valid_probability=True,
        )
        probability_matrix[grid_index] = metrics.pop("valid_probability")
        metrics.pop("full_probability")

        rows.append({
            "anchor_id": anchor_id,
            "source_row_index": anchor_row["source_row_index"],
            "theta_index": theta_index,
            "parameter_name": parameter_name,
            "grid_index": int(grid_index),
            "theta_value": float(value),
            "theta_over_period": float(value / period),
            "theta_over_pi": float(value / np.pi),
            "period": period,
            "anchor_theta_raw": original_raw,
            "anchor_theta_canonical": original_canonical,
            "is_original_theta": is_original,
            "distance_to_anchor": circular_distance(value, original_raw, period),
            **metrics,
            "optimizer_used": False,
        })

    task_df = pd.DataFrame(rows)

    original_rows = task_df.loc[task_df["is_original_theta"]]
    if len(original_rows) != 1:
        raise RuntimeError(
            f"Esperado um ponto original para theta_{theta_index}, âncora {anchor_id}; "
            f"encontrados {len(original_rows)}."
        )
    original_p = float(original_rows.iloc[0]["p_optimal"])
    if not np.isclose(original_p, anchor_metrics["p_optimal"], atol=1e-12, rtol=1e-10):
        raise RuntimeError(
            f"O ponto original de theta_{theta_index}, âncora {anchor_id}, não reproduziu "
            f"P(X_opt): {original_p} versus {anchor_metrics['p_optimal']}"
        )

    p_min = float(task_df["p_optimal"].min())
    p_max = float(task_df["p_optimal"].max())
    span = p_max - p_min
    flat_threshold = max(
        SWEEP_FLAT_ABS_TOL,
        SWEEP_FLAT_REL_TOL * max(abs(p_max), 1.0),
    )
    task_df["p_span"] = span
    task_df["flat_threshold"] = flat_threshold
    task_df["is_flat_sweep"] = bool(span <= flat_threshold)

    task_df.to_pickle(summary_path)
    np.savez_compressed(
        distribution_path,
        valid_probability=probability_matrix,
        theta_grid=grid,
        valid_bitstrings=valid_bitstrings,
        valid_objectives=valid_objectives,
    )
    return task_df, distribution_path


if len(anchors_df) != N_ANCHORS:
    raise RuntimeError(
        f"A campanha exige {N_ANCHORS} vetores selecionados; encontrados {len(anchors_df)}."
    )

for theta_index in DETAILED_THETA_INDICES:
    if not 0 <= theta_index < N_PARAMETERS:
        raise IndexError(f"theta_{theta_index} não existe no circuito com {N_PARAMETERS} parâmetros.")

all_detailed_frames = []
detailed_distribution_paths = {}
total_tasks = len(anchors_df) * len(DETAILED_THETA_INDICES)
completed_tasks = 0

for _, anchor_row in anchors_df.iterrows():
    anchor_id = int(anchor_row["anchor_id"])
    for theta_index in DETAILED_THETA_INDICES:
        task_df, distribution_path = run_single_parameter_task(anchor_row, theta_index)
        all_detailed_frames.append(task_df)
        detailed_distribution_paths[(anchor_id, int(theta_index))] = distribution_path
        completed_tasks += 1
        if completed_tasks == 1 or completed_tasks % 25 == 0 or completed_tasks == total_tasks:
            print(
                f"tarefas concluídas: {completed_tasks}/{total_tasks} | "
                f"anchor={anchor_id:03d} | theta_{theta_index}"
            )

individual_sweep_df = pd.concat(all_detailed_frames, ignore_index=True)
individual_sweep_path = TABLE_DIR / "individual_sweeps_100_vectors.pkl"
individual_sweep_df.to_pickle(individual_sweep_path)

print("Âncoras avaliadas:", individual_sweep_df["anchor_id"].nunique())
print("Thetas avaliados:", sorted(individual_sweep_df["theta_index"].unique().tolist()))
print("Total de avaliações detalhadas:", len(individual_sweep_df))
print("Tabela salva em:", individual_sweep_path.resolve())

### Célula 14 — Distribuição das 100 curvas, valores originais e auditorias

Esta célula produz três visualizações complementares:

1. **Sobreposição por parâmetro:** cada painel contém as 100 curvas brutas de um mesmo $\theta_j$, os 100 pontos originais e o envelope entre os percentis 5% e 95%.
2. **Valores originais reunidos:** um único gráfico mostra, para cada $\theta_j$, os 100 valores canônicos provenientes dos 100 vetores completos.
3. **Mapa de distribuição:** para cada parâmetro, uma matriz mostra a probabilidade em função da fase da varredura e do vetor.

Correções aplicadas:

- o eixo vertical é limitado a $[-0.02,1.02]$;
- o rótulo passa a ser $P(\mathcal{X}_{\mathrm{opt}})$, pois pode existir mais de um bitstring ótimo;
- valores brutos e canônicos são mantidos nas tabelas;
- o mapeamento de qubits e ativos é auditado como constante para cada índice;
- parâmetros `CRY` recebem uma comparação explícita entre $[0,2\pi]$ e $[2\pi,4\pi]$;
- `theta_17` e `theta_25` continuam sendo comparados em cada um dos 100 vetores, sem normalização de probabilidade.

In [ ]:
# ============================================================
# 14. 100 CURVAS, DISTRIBUIÇÕES, PONTOS ORIGINAIS E AUDITORIAS
# ============================================================

if individual_sweep_df["anchor_id"].nunique() != N_ANCHORS:
    raise RuntimeError(
        f"Esperados {N_ANCHORS} vetores no resultado; encontrados "
        f"{individual_sweep_df['anchor_id'].nunique()}."
    )

parameter_order = list(ansatz.parameters)
if len(parameter_order) != N_PARAMETERS:
    raise RuntimeError("A ordem de parâmetros do ansatz está inconsistente.")

# -----------------------------------------------------------------
# 14.1 Mapeamento fixo theta -> bloco -> qubits -> ativos
# -----------------------------------------------------------------
selected_parameter_map_df = parameter_map_df.loc[
    parameter_map_df["theta_index"].isin(DETAILED_THETA_INDICES)
].copy().sort_values("theta_index")
selected_parameter_map_df["parameter_name_from_ansatz"] = selected_parameter_map_df[
    "theta_index"
].map(lambda index: str(parameter_order[int(index)]))

if selected_parameter_map_df["parameter_name_from_ansatz"].duplicated().any():
    duplicates = selected_parameter_map_df.loc[
        selected_parameter_map_df["parameter_name_from_ansatz"].duplicated(False),
        ["theta_index", "parameter_name_from_ansatz"],
    ]
    raise RuntimeError(f"Objetos de parâmetro duplicados encontrados:\n{duplicates}")

mapping_audit_df = selected_parameter_map_df[[
    "theta_index",
    "parameter_name_from_ansatz",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "angular_period",
]].copy()
mapping_audit_df["period_over_pi"] = mapping_audit_df["angular_period"] / np.pi
mapping_audit_df["mapping_depends_on_anchor"] = False

print("MAPEAMENTO FIXO: os ativos dependem do theta/bloco, não do vetor selecionado.")
display(mapping_audit_df)
mapping_audit_df.to_csv(TABLE_DIR / "fixed_theta_asset_mapping.csv", index=False)

# Um e somente um ponto original para cada combinação vetor x theta.
original_points_df = individual_sweep_df.loc[
    individual_sweep_df["is_original_theta"]
].copy()
expected_original_points = N_ANCHORS * len(DETAILED_THETA_INDICES)
if len(original_points_df) != expected_original_points:
    raise RuntimeError(
        f"Esperados {expected_original_points} pontos originais; "
        f"encontrados {len(original_points_df)}."
    )
if original_points_df.duplicated(["anchor_id", "theta_index"]).any():
    raise RuntimeError("Existem pontos originais duplicados para o mesmo vetor e theta.")

original_points_df = original_points_df.merge(
    selected_parameter_map_df[[
        "theta_index", "ansatz_gate_type", "primitive_physical_type",
        "logical_block_qubits", "logical_assets", "angular_period",
    ]],
    on="theta_index",
    how="left",
    validate="many_to_one",
)

# Resumo que mostra explicitamente a dispersão dos 100 valores originais.
original_value_summary_df = original_points_df.groupby("theta_index").agg(
    n_vectors=("anchor_id", "nunique"),
    period=("period", "first"),
    theta_raw_min=("anchor_theta_raw", "min"),
    theta_raw_max=("anchor_theta_raw", "max"),
    theta_raw_mean=("anchor_theta_raw", "mean"),
    theta_raw_std=("anchor_theta_raw", "std"),
    theta_canonical_min=("anchor_theta_canonical", "min"),
    theta_canonical_max=("anchor_theta_canonical", "max"),
    theta_canonical_mean=("anchor_theta_canonical", "mean"),
    theta_canonical_std=("anchor_theta_canonical", "std"),
    p_original_min=("p_optimal", "min"),
    p_original_max=("p_optimal", "max"),
    p_original_mean=("p_optimal", "mean"),
).reset_index()
original_value_summary_df["n_unique_canonical_8dp"] = original_value_summary_df[
    "theta_index"
].map(
    lambda theta_index: original_points_df.loc[
        original_points_df["theta_index"].eq(theta_index),
        "anchor_theta_canonical",
    ].round(8).nunique()
)

display(original_value_summary_df)
original_value_summary_df.to_csv(
    TABLE_DIR / "original_theta_values_100_vectors_summary.csv", index=False
)
original_points_df.to_pickle(TABLE_DIR / "original_theta_points_100_vectors.pkl")


def common_curve_matrix(theta_index, n_points=COMMON_PHASE_POINTS):
    """Interpola as curvas de um theta em uma fase comum 0..1."""
    theta_index = int(theta_index)
    common_phase = np.linspace(0.0, 1.0, int(n_points))
    curves = []
    anchor_ids = []
    for anchor_id, group in individual_sweep_df.loc[
        individual_sweep_df["theta_index"].eq(theta_index)
    ].groupby("anchor_id", sort=True):
        group = group.sort_values("theta_over_period")
        curves.append(np.interp(
            common_phase,
            group["theta_over_period"].to_numpy(dtype=float),
            group["p_optimal"].to_numpy(dtype=float),
        ))
        anchor_ids.append(int(anchor_id))
    matrix = np.asarray(curves, dtype=float)
    if matrix.shape != (N_ANCHORS, int(n_points)):
        raise RuntimeError(
            f"Matriz inesperada para theta_{theta_index}: {matrix.shape}; "
            f"esperado {(N_ANCHORS, int(n_points))}."
        )
    return common_phase, np.asarray(anchor_ids, dtype=int), matrix


# -----------------------------------------------------------------
# 14.2 FIGURA 1 — 100 curvas sobrepostas em cada theta
# -----------------------------------------------------------------
n_theta = len(DETAILED_THETA_INDICES)
n_cols = 3
n_rows = int(np.ceil(n_theta / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 4.8 * n_rows), squeeze=False)
axes_flat = axes.ravel()
curve_diversity_rows = []

for panel_index, theta_index in enumerate(DETAILED_THETA_INDICES):
    ax = axes_flat[panel_index]
    map_row = parameter_map_df.loc[
        parameter_map_df["theta_index"].eq(theta_index)
    ].iloc[0]
    period = float(map_row["angular_period"])
    common_phase, anchor_ids, curve_matrix = common_curve_matrix(theta_index)
    common_angle = common_phase * period

    # As 100 curvas são desenhadas sem normalização de P.
    for curve in curve_matrix:
        ax.plot(common_angle, curve, linewidth=0.75, alpha=0.12)

    q05, q50, q95 = np.quantile(curve_matrix, [0.05, 0.50, 0.95], axis=0)
    ax.fill_between(common_angle, q05, q95, alpha=0.20, label="faixa 5%–95%")
    ax.plot(common_angle, q50, linewidth=2.2, label="mediana das 100 curvas")

    original_group = original_points_df.loc[
        original_points_df["theta_index"].eq(theta_index)
    ]
    ax.scatter(
        original_group["anchor_theta_canonical"],
        original_group["p_optimal"],
        s=16,
        alpha=0.55,
        zorder=5,
        label="100 valores originais",
    )

    if np.isclose(period, 4 * np.pi):
        ax.axvline(2 * np.pi, linestyle="--", linewidth=1.0, alpha=0.7)
        ticks = [0, np.pi, 2*np.pi, 3*np.pi, 4*np.pi]
        tick_labels = ["0", "$\\pi$", "$2\\pi$", "$3\\pi$", "$4\\pi$"]
    else:
        ticks = [0, 0.5*np.pi, np.pi, 1.5*np.pi, 2*np.pi]
        tick_labels = ["0", "$\\pi/2$", "$\\pi$", "$3\\pi/2$", "$2\\pi$"]

    ax.set_xticks(ticks, tick_labels)
    ax.set_xlim(0.0, period)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel(f"valor de theta_{theta_index} (rad)")
    ax.set_ylabel(r"$P(\mathcal{X}_{\mathrm{opt}})$")
    ax.set_title(
        f"theta_{theta_index} | {map_row['ansatz_gate_type']}/{map_row['primitive_physical_type']} "
        f"| período={period/np.pi:.0f}π\n"
        f"qubits={map_row['logical_block_qubits']} | ativos fixos={map_row['logical_assets']}"
    )
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)

    pointwise_std = np.std(curve_matrix, axis=0)
    curve_diversity_rows.append({
        "theta_index": int(theta_index),
        "n_vectors": int(curve_matrix.shape[0]),
        "period": period,
        "mean_pointwise_std_P": float(np.mean(pointwise_std)),
        "max_pointwise_std_P": float(np.max(pointwise_std)),
        "mean_90pct_envelope_width": float(np.mean(q95 - q05)),
        "max_90pct_envelope_width": float(np.max(q95 - q05)),
        "n_flat_curves": int(
            individual_sweep_df.loc[
                individual_sweep_df["theta_index"].eq(theta_index)
            ].groupby("anchor_id")["is_flat_sweep"].first().sum()
        ),
    })

for unused_index in range(n_theta, len(axes_flat)):
    axes_flat[unused_index].axis("off")

fig.suptitle(
    "Varredura individual em 100 vetores mascarados — curvas brutas, envelope e pontos originais",
    fontsize=15,
    y=1.002,
)
fig.tight_layout()
overlay_path = FIGURE_DIR / "100_masked_vectors_raw_sweeps_overlay.png"
fig.savefig(overlay_path, dpi=180, bbox_inches="tight")
plt.show()
print("Figura salva em:", overlay_path.resolve())

curve_diversity_df = pd.DataFrame(curve_diversity_rows)
display(curve_diversity_df)
curve_diversity_df.to_csv(TABLE_DIR / "curve_diversity_100_vectors.csv", index=False)

# -----------------------------------------------------------------
# 14.3 FIGURA 2 — todos os valores originais em um único gráfico
# -----------------------------------------------------------------
ordered_thetas = DETAILED_THETA_INDICES
positions = np.arange(len(ordered_thetas), dtype=float)
canonical_groups = [
    original_points_df.loc[
        original_points_df["theta_index"].eq(theta_index),
        "anchor_theta_canonical",
    ].to_numpy(dtype=float)
    for theta_index in ordered_thetas
]

fig, ax = plt.subplots(figsize=(15, 7))
ax.boxplot(
    canonical_groups,
    positions=positions,
    widths=0.50,
    showfliers=False,
    medianprops={"linewidth": 2.0},
)

rng = np.random.default_rng(RANDOM_SEED)
for position, theta_index, values in zip(positions, ordered_thetas, canonical_groups):
    jitter = rng.normal(0.0, 0.055, size=len(values))
    ax.scatter(
        np.full(len(values), position) + jitter,
        values,
        s=22,
        alpha=0.55,
    )

ax.set_xticks(positions, [f"theta_{theta_index}" for theta_index in ordered_thetas])
ax.set_yticks(
    [0, np.pi, 2*np.pi, 3*np.pi, 4*np.pi],
    ["0", "$\\pi$", "$2\\pi$", "$3\\pi$", "$4\\pi$"],
)
ax.set_ylim(-0.15, 4*np.pi + 0.15)
ax.set_xlabel("índice do parâmetro")
ax.set_ylabel("valor original canônico (rad)")
ax.set_title(
    "Distribuição dos valores originais de cada theta nos 100 vetores selecionados\n"
    "cada ponto corresponde a um vetor completo diferente"
)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
original_values_figure_path = FIGURE_DIR / "original_theta_values_100_vectors_combined.png"
fig.savefig(original_values_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Figura salva em:", original_values_figure_path.resolve())

# -----------------------------------------------------------------
# 14.4 FIGURA 3 — mapa de distribuição das 100 curvas por theta
# -----------------------------------------------------------------
fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 4.6 * n_rows), squeeze=False)
axes_flat = axes.ravel()
last_image = None

for panel_index, theta_index in enumerate(DETAILED_THETA_INDICES):
    ax = axes_flat[panel_index]
    common_phase, anchor_ids, curve_matrix = common_curve_matrix(theta_index)

    # Ordena as linhas pelo valor original canônico para revelar padrões.
    original_order = original_points_df.loc[
        original_points_df["theta_index"].eq(theta_index),
        ["anchor_id", "anchor_theta_canonical"],
    ].sort_values("anchor_theta_canonical")["anchor_id"].to_numpy(dtype=int)
    row_lookup = {anchor_id: row for row, anchor_id in enumerate(anchor_ids)}
    ordered_matrix = curve_matrix[[row_lookup[int(a)] for a in original_order]]

    last_image = ax.imshow(
        ordered_matrix,
        aspect="auto",
        origin="lower",
        extent=[0.0, 1.0, 0, N_ANCHORS - 1],
        vmin=0.0,
        vmax=1.0,
        interpolation="nearest",
    )
    ax.set_xlabel(r"fase da varredura $\theta_j/T_j$")
    ax.set_ylabel("vetores ordenados")
    ax.set_title(f"theta_{theta_index} | 100 curvas")

for unused_index in range(n_theta, len(axes_flat)):
    axes_flat[unused_index].axis("off")

fig.suptitle(
    r"Distribuição de $P(\mathcal{X}_{\mathrm{opt}})$ nos 100 vetores — linhas ordenadas pelo theta original",
    fontsize=15,
    y=1.002,
)
if last_image is not None:
    fig.colorbar(last_image, ax=axes_flat[:n_theta].tolist(), shrink=0.75, label=r"$P(\mathcal{X}_{\mathrm{opt}})$")
fig.subplots_adjust(top=0.94, wspace=0.25, hspace=0.32)
heatmap_path = FIGURE_DIR / "100_vectors_probability_distribution_heatmaps.png"
fig.savefig(heatmap_path, dpi=180, bbox_inches="tight")
plt.show()
print("Figura salva em:", heatmap_path.resolve())

# -----------------------------------------------------------------
# 14.5 Auditoria empírica: CRY precisa de 4pi para esta probabilidade?
# -----------------------------------------------------------------
periodicity_rows = []
cry_thetas = selected_parameter_map_df.loc[
    selected_parameter_map_df["primitive_physical_type"].eq("CRY"),
    "theta_index",
].astype(int).tolist()

for theta_index in cry_thetas:
    for anchor_id, group in individual_sweep_df.loc[
        individual_sweep_df["theta_index"].eq(theta_index)
    ].groupby("anchor_id", sort=True):
        group = group.sort_values("theta_value")
        period = float(group["period"].iloc[0])
        if not np.isclose(period, 4 * np.pi):
            raise RuntimeError(f"theta_{theta_index} é CRY, mas período={period}.")

        half_grid = np.linspace(0.0, 2*np.pi, COMMON_PHASE_POINTS)
        first_cycle = np.interp(
            half_grid,
            group["theta_value"],
            group["p_optimal"],
        )
        second_cycle = np.interp(
            half_grid + 2*np.pi,
            group["theta_value"],
            group["p_optimal"],
        )
        max_diff = float(np.max(np.abs(first_cycle - second_cycle)))
        rms_diff = float(np.sqrt(np.mean((first_cycle - second_cycle) ** 2)))
        periodicity_rows.append({
            "anchor_id": int(anchor_id),
            "theta_index": int(theta_index),
            "max_abs_difference_P_between_cycles": max_diff,
            "rms_difference_P_between_cycles": rms_diff,
            "probability_is_effectively_2pi_periodic": bool(
                np.allclose(
                    first_cycle,
                    second_cycle,
                    atol=PERIODICITY_ATOL,
                    rtol=PERIODICITY_RTOL,
                )
            ),
        })

cry_periodicity_per_vector_df = pd.DataFrame(periodicity_rows)
cry_periodicity_summary_df = cry_periodicity_per_vector_df.groupby("theta_index").agg(
    n_vectors=("anchor_id", "nunique"),
    max_difference_over_100=("max_abs_difference_P_between_cycles", "max"),
    median_difference_over_100=("max_abs_difference_P_between_cycles", "median"),
    mean_rms_difference=("rms_difference_P_between_cycles", "mean"),
    n_effectively_2pi=("probability_is_effectively_2pi_periodic", "sum"),
).reset_index()
cry_periodicity_summary_df["all_100_effectively_2pi"] = (
    cry_periodicity_summary_df["n_effectively_2pi"] == N_ANCHORS
)

display(cry_periodicity_summary_df)
cry_periodicity_per_vector_df.to_csv(
    TABLE_DIR / "cry_2pi_vs_4pi_periodicity_per_vector.csv", index=False
)
cry_periodicity_summary_df.to_csv(
    TABLE_DIR / "cry_2pi_vs_4pi_periodicity_summary.csv", index=False
)

# -----------------------------------------------------------------
# 14.6 Auditoria theta_17 x theta_25 em cada um dos 100 vetores
# -----------------------------------------------------------------
pair_rows = []
for anchor_id in sorted(individual_sweep_df["anchor_id"].unique()):
    group_17 = individual_sweep_df.loc[
        individual_sweep_df["anchor_id"].eq(anchor_id)
        & individual_sweep_df["theta_index"].eq(17)
    ].sort_values("theta_over_period")
    group_25 = individual_sweep_df.loc[
        individual_sweep_df["anchor_id"].eq(anchor_id)
        & individual_sweep_df["theta_index"].eq(25)
    ].sort_values("theta_over_period")

    common_phase = np.linspace(0.0, 1.0, COMMON_PHASE_POINTS)
    p17 = np.interp(common_phase, group_17["theta_over_period"], group_17["p_optimal"])
    p25 = np.interp(common_phase, group_25["theta_over_period"], group_25["p_optimal"])
    original_17 = group_17.loc[group_17["is_original_theta"]].iloc[0]
    original_25 = group_25.loc[group_25["is_original_theta"]].iloc[0]

    pair_rows.append({
        "anchor_id": int(anchor_id),
        "theta_17_original_raw": float(original_17["anchor_theta_raw"]),
        "theta_25_original_raw": float(original_25["anchor_theta_raw"]),
        "theta_17_original_canonical": float(original_17["anchor_theta_canonical"]),
        "theta_25_original_canonical": float(original_25["anchor_theta_canonical"]),
        "original_value_difference_abs": float(abs(
            original_17["anchor_theta_canonical"] - original_25["anchor_theta_canonical"]
        )),
        "max_abs_curve_difference_common_phase": float(np.max(np.abs(p17 - p25))),
        "rms_curve_difference_common_phase": float(np.sqrt(np.mean((p17 - p25) ** 2))),
        "curves_equal_common_phase": bool(
            np.allclose(p17, p25, atol=1e-10, rtol=1e-8)
        ),
    })

theta_17_25_per_vector_df = pd.DataFrame(pair_rows)
theta_17_25_summary_df = pd.DataFrame([{
    "n_vectors": int(len(theta_17_25_per_vector_df)),
    "n_equal_curves": int(theta_17_25_per_vector_df["curves_equal_common_phase"].sum()),
    "n_different_curves": int((~theta_17_25_per_vector_df["curves_equal_common_phase"]).sum()),
    "min_original_value_difference_abs": float(
        theta_17_25_per_vector_df["original_value_difference_abs"].min()
    ),
    "max_original_value_difference_abs": float(
        theta_17_25_per_vector_df["original_value_difference_abs"].max()
    ),
    "max_curve_difference_over_100": float(
        theta_17_25_per_vector_df["max_abs_curve_difference_common_phase"].max()
    ),
    "median_curve_difference_over_100": float(
        theta_17_25_per_vector_df["max_abs_curve_difference_common_phase"].median()
    ),
}])

display(theta_17_25_summary_df)
theta_17_25_per_vector_df.to_csv(
    TABLE_DIR / "theta_17_vs_theta_25_per_vector.csv", index=False
)
theta_17_25_summary_df.to_csv(
    TABLE_DIR / "theta_17_vs_theta_25_summary.csv", index=False
)

print("\nINTERPRETAÇÃO AUTOMÁTICA")
for _, row in cry_periodicity_summary_df.iterrows():
    theta_index = int(row["theta_index"])
    if bool(row["all_100_effectively_2pi"]):
        print(
            f"- theta_{theta_index}: a unidade CRY foi varrida até 4π, mas "
            "P(X_opt) repetiu após 2π nos 100 vetores."
        )
    else:
        print(
            f"- theta_{theta_index}: pelo menos um vetor distinguiu o primeiro "
            "e o segundo ciclo; manter 4π é necessário para este observável."
        )

print(
    "- Os ativos exibidos em cada painel são fixos para aquele theta. "
    "Eles diferem entre índices porque cada parâmetro pertence a outro bloco do ansatz."
)
print(
    "- O gráfico de pontos/boxplots reúne os 100 valores originais e permite verificar "
    "diretamente se cada theta assume valores diferentes entre os vetores."
)